# Thai Customer Chat — Topic Modeling

**Stack:** `multilingual-e5-large` (embedding) + `BERTopic` (clustering) + `Typhoon via Ollama` (labeling)

**Runtime:** เลือก `Runtime → Change runtime type → T4 GPU` ก่อน Run

---
### Pipeline
```
[1] Load Data  →  [2] Preprocess  →  [3] Embed  →  [4] BERTopic  →  [5] Label  →  [6] Report
```

## ⚙️ Setup — Install dependencies

In [ ]:
# ── Install all required packages ─────────────────────────────────────────────
# (~3-5 minutes on Colab first run)
!pip install -q \
    bertopic>=0.16.0 \
    sentence-transformers>=2.7.0 \
    torch>=2.0.0 \
    pythainlp>=5.0.0 \
    umap-learn>=0.5.6 \
    hdbscan>=0.8.33 \
    ollama>=0.3.0 \
    pandas>=2.0.0 \
    numpy>=1.24.0 \
    scikit-learn>=1.3.0 \
    openpyxl>=3.1.0 \
    matplotlib>=3.7.0 \
    seaborn>=0.12.0 \
    joblib>=1.3.0

print("✓ Packages installed")

In [ ]:
# ── Install Ollama binary & start server ──────────────────────────────────────
import subprocess
import time
import os

print("Installing Ollama...")
os.system("curl -fsSL https://ollama.com/install.sh | sh")

print("Starting Ollama server in background...")
_ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)  # Wait for server to be ready
print("✓ Ollama server running")

In [ ]:
# ── Pull Typhoon model (~2.6 GB — takes 3-5 min on Colab) ─────────────────────
# ตัวเลือก:
#   scb10x/typhoon2.1-gemma3-4b   (2.6 GB) ← ใช้ไฟล์นี้เป็น default
#   scb10x/typhoon2.5-qwen3-4b    (2.5 GB) ← รุ่นใหม่กว่า

OLLAMA_MODEL = "scb10x/typhoon2.1-gemma3-4b"  # เปลี่ยนได้

result = subprocess.run(
    ["ollama", "pull", OLLAMA_MODEL],
    capture_output=False,
    text=True
)
if result.returncode == 0:
    print(f"✓ Model ready: {OLLAMA_MODEL}")
else:
    print("✗ Pull failed — check model name with: !ollama list")

## 📁 Upload Data

เลือกวิธีใดวิธีหนึ่ง:
- **Option A** — Upload ไฟล์ CSV/Excel โดยตรง
- **Option B** — Mount Google Drive แล้วระบุ path

In [ ]:
# ── Option A: Upload ไฟล์โดยตรง ───────────────────────────────────────────────
from google.colab import files
import os

os.makedirs("/content/data/raw", exist_ok=True)

print("เลือกไฟล์ chat_data.csv หรือ .xlsx")
uploaded = files.upload()

for fname in uploaded:
    dest = f"/content/data/raw/{fname}"
    with open(dest, "wb") as f:
        f.write(uploaded[fname])
    print(f"✓ Saved: {dest}")
    FILE_PATH = dest

In [ ]:
# ── Option B: Mount Google Drive ──────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# FILE_PATH = "/content/drive/MyDrive/YOUR_FOLDER/chat_data.csv"  # ← แก้ path

## ⚙️ Configuration — แก้ที่นี่เท่านั้น

In [ ]:
# ═══════════════════════════════════════════════════════════
#   USER CONFIGURATION — แก้ที่นี่เท่านั้น
# ═══════════════════════════════════════════════════════════

# FILE_PATH ถูก set จาก cell Upload ด้านบนแล้ว
# หากไม่ได้ Upload ให้ uncomment บรรทัดด้านล่างและแก้ path
# FILE_PATH = "/content/data/raw/chat_data.csv"

MESSAGE_COL    = "message"     # ชื่อ column ที่เก็บข้อความ
ROLE_COL       = None          # ชื่อ column ที่ระบุ sender (None ถ้าไม่มี)
CUSTOMER_VALUE = None          # ค่าที่แทน customer เช่น "customer" (None ถ้าไม่มี)
SESSION_COL    = "session_id"  # ชื่อ column session (None ถ้าไม่มี)
TOP_N_CHART    = 15            # จำนวน topics ที่แสดงในกราฟ

# ── Model Config ──────────────────────────────────────────
EMBEDDING_MODEL    = "intfloat/multilingual-e5-large"
EMBED_BATCH_SIZE   = 32        # RTX / T4: 32 safe, เพิ่มเป็น 64 ถ้า VRAM พอ
# OLLAMA_MODEL ถูก set ใน cell Install Ollama ด้านบนแล้ว
OLLAMA_HOST        = "http://localhost:11434"
TEXT_DELIMITER     = "####"
MAX_DOC_CHARS      = 300

# ── Output Paths ──────────────────────────────────────────
import os
for d in ["/content/data/processed", "/content/data/embeddings",
           "/content/data/topics", "/content/models", "/content/outputs"]:
    os.makedirs(d, exist_ok=True)

print("✓ Config ready")
print(f"  File       : {FILE_PATH}")
print(f"  Message col: {MESSAGE_COL}")
print(f"  LLM model  : {OLLAMA_MODEL}")

## [1/6] Load Data

In [ ]:
import pandas as pd
import logging

logging.basicConfig(format="%(asctime)s | %(levelname)s | %(message)s", level=logging.INFO)


def load_chat_data(file_path, message_col, role_col=None, customer_value=None, session_col=None):
    logging.info(f"Loading: {file_path}")
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path, encoding="utf-8-sig")
    elif file_path.endswith((".xlsx", ".xls")):
        df = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported format. Use .csv, .xlsx, or .xls")

    logging.info(f"Loaded {len(df):,} rows | Columns: {list(df.columns)}")

    if role_col and customer_value:
        before = len(df)
        df = df[df[role_col] == customer_value].copy()
        logging.info(f"Customer filter: {before:,} → {len(df):,} rows")

    df = df.dropna(subset=[message_col])
    df = df[df[message_col].astype(str).str.strip() != ""]
    df = df.rename(columns={message_col: "raw_message"})
    if session_col and session_col in df.columns:
        df = df.rename(columns={session_col: "session_id"})
    df = df.reset_index(drop=True)
    logging.info(f"Final: {len(df):,} customer messages ready")
    return df


# ── Run ───────────────────────────────────────────────────────────────────────
df = load_chat_data(
    file_path=FILE_PATH,
    message_col=MESSAGE_COL,
    role_col=ROLE_COL,
    customer_value=CUSTOMER_VALUE,
    session_col=SESSION_COL,
)

print("\n=== Data Preview ===")
print(df[["raw_message"]].head(5).to_string())
print(f"\nTotal messages : {len(df):,}")
print(f"Avg length     : {df['raw_message'].str.len().mean():.0f} chars")

## [2/6] Preprocess (Thai-English mixed text)

In [ ]:
import re
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_stopwords
from pythainlp.util import normalize as thai_normalize

# ── Stop words ────────────────────────────────────────────────────────────────
THAI_STOPS = set(thai_stopwords())
CHAT_STOPS = {
    "ครับ", "ค่ะ", "คะ", "นะ", "นะครับ", "นะคะ", "ด้วยนะ",
    "ขอบคุณ", "ขอบคุณครับ", "ขอบคุณค่ะ", "สวัสดี", "สวัสดีครับ",
    "โอเค", "ok", "okay",
    "อยาก", "ต้องการ", "ขอ", "ได้", "มี", "ไม่",
    "ถาม", "สอบถาม", "รบกวน", "ช่วย", "หน่อย",
}
ALL_THAI_STOPS = THAI_STOPS.union(CHAT_STOPS)
ENGLISH_STOPS = {
    "i", "me", "my", "we", "you", "he", "she", "it", "they",
    "is", "am", "are", "was", "were", "be", "been", "have", "has",
    "do", "does", "did", "will", "would", "could", "should",
    "the", "a", "an", "and", "or", "but", "in", "on", "at",
    "to", "for", "of", "with", "hi", "hello", "thanks", "thank",
    "please", "yes", "no", "ok", "okay",
}


def detect_lang(text):
    thai = len(re.findall(r'[\u0E00-\u0E7F]', text))
    eng  = len(re.findall(r'[a-zA-Z]', text))
    tot  = thai + eng
    if tot == 0:
        return "other"
    r = thai / tot
    return "th" if r >= 0.8 else ("en" if r <= 0.2 else "mixed")


def normalize(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    text = thai_normalize(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\b0\d{8,9}\b", " ", text)
    text = re.sub(r"[\w.-]+@[\w.-]+\.\w+", " ", text)
    text = re.sub(r"[\U00010000-\U0010ffff]", " ", text, flags=re.UNICODE)
    text = re.sub(r"[\U0001F600-\U0001F64F]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r'[a-zA-Z]+', lambda m: m.group().lower(), text)
    text = re.sub(r"[^\u0E00-\u0E7Fa-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    if not text:
        return []
    tokens = []
    for seg in re.split(r'(\s+)', text):
        seg = seg.strip()
        if not seg:
            continue
        if re.search(r'[\u0E00-\u0E7F]', seg):
            tokens.extend(word_tokenize(seg, engine="newmm", keep_whitespace=False))
        elif re.search(r'[a-zA-Z]', seg):
            tokens.extend(seg.split())
    return tokens


def remove_stops(tokens):
    return [t for t in tokens
            if t not in ALL_THAI_STOPS
            and t not in ENGLISH_STOPS
            and len(t) >= 2]


def preprocess_one(text):
    return " ".join(remove_stops(tokenize(normalize(text))))


# ── Run ───────────────────────────────────────────────────────────────────────
logging.info("Detecting language per message...")
df["lang_type"] = df["raw_message"].apply(detect_lang)
logging.info(f"Language distribution:\n{df['lang_type'].value_counts().to_string()}")

logging.info("Preprocessing messages...")
df["text_clean"] = df["raw_message"].apply(preprocess_one)
df["char_count"] = df["raw_message"].str.len()

before = len(df)
df = df[df["text_clean"].str.len() >= 5].reset_index(drop=True)
logging.info(f"Removed short messages: {before:,} → {len(df):,} rows")

df.to_csv("/content/data/processed/messages_clean.csv", index=False, encoding="utf-8-sig")
print(f"\n✓ Preprocessed {len(df):,} messages")
df[["raw_message", "text_clean", "lang_type"]].head(3)

## [3/6] Generate Embeddings (multilingual-e5-large)

In [ ]:
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer

formatted_dt = datetime.now().strftime("%d_%b_%Y_%H_%M_%S")

logging.info(f"Loading embedding model: {EMBEDDING_MODEL}")
_embed_model = SentenceTransformer(EMBEDDING_MODEL)
logging.info("Embedding model ready.")


def embed_texts(texts):
    """multilingual-e5-large requires 'passage: ' prefix for document embedding."""
    prefixed = [f"passage: {t}" for t in texts]
    return _embed_model.encode(
        prefixed,
        batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )


# ── Run ───────────────────────────────────────────────────────────────────────
texts = df["text_clean"].astype(str).tolist()
n     = len(texts)
logging.info(f"Generating embeddings for {n:,} messages...")

all_embs = []
for start in range(0, n, EMBED_BATCH_SIZE):
    end   = min(start + EMBED_BATCH_SIZE, n)
    batch = embed_texts(texts[start:end])
    all_embs.append(batch)

    processed = end
    if processed % 1000 == 0 or processed == n:
        checkpoint = np.vstack(all_embs)
        np.save(f"/content/data/embeddings/checkpoint_{processed}_{formatted_dt}.npy", checkpoint)
        logging.info(f"Checkpoint: {processed:,}/{n:,} rows saved")

all_embs_array = np.vstack(all_embs)
np.save(f"/content/data/embeddings/full_{formatted_dt}.npy", all_embs_array)

df["embedding"] = list(all_embs_array)
logging.info(f"✓ Embedding complete — shape: {all_embs_array.shape}")
print(f"✓ Embeddings: {all_embs_array.shape}")

## [4/6] BERTopic Clustering

In [ ]:
import joblib
from bertopic import BERTopic

emb_array = np.array(df["embedding"].tolist())
docs      = df["text_clean"].astype(str).tolist()

# Reuse same model — no extra download
sentence_model = _embed_model

n        = len(df)
min_size = 5 if n < 1000 else (10 if n < 5000 else 20)
logging.info(f"Fitting BERTopic | {n:,} messages | min_topic_size={min_size}")

bertopic_model = BERTopic(
    embedding_model=sentence_model,
    language="multilingual",
    calculate_probabilities=True,
    min_topic_size=min_size,
    nr_topics="auto",
    verbose=True,
)

topics, probs = bertopic_model.fit_transform(docs, emb_array)
df["topic"]   = topics

# probs is 2-D (n_docs, n_topics) — store max probability per document
if probs is not None and np.ndim(probs) == 2:
    df["topic_prob"] = probs.max(axis=1)
else:
    df["topic_prob"] = probs

# Reduce outliers only if topic -1 exists
if -1 in topics:
    new_topics = bertopic_model.reduce_outliers(docs, topics, strategy="distributions")
else:
    new_topics = topics
df["topic_final"] = new_topics

# Save model
joblib.dump(bertopic_model, f"/content/models/bertopic_{formatted_dt}.joblib")
df.to_csv(f"/content/data/topics/raw_clusters_{formatted_dt}.csv", index=False, encoding="utf-8-sig")

rep_docs = bertopic_model.get_representative_docs()
n_topics = len(set(new_topics)) - (1 if -1 in new_topics else 0)
logging.info(f"✓ Topics identified: {n_topics}")
print(f"✓ Topics: {n_topics}")
bertopic_model.get_topic_info().head(10)

## [5/6] Label Topics with Typhoon (local Ollama)

In [ ]:
import json
import time
import ollama


def build_prompt(docs_str):
    return f"""คุณกำลังวิเคราะห์ข้อความของลูกค้าจาก Chat ของธุรกิจไทย
ข้อความเหล่านี้เป็นภาษาไทย ภาษาอังกฤษ หรือผสมกัน

ข้อความตัวแทนของกลุ่มนี้ คั่นด้วย {TEXT_DELIMITER}:
{docs_str}

วิเคราะห์ว่าลูกค้ากลุ่มนี้ต้องการอะไร หรือมีปัญหาเรื่องอะไร

กฎ:
- ชื่อหัวข้อ: 3-6 คำภาษาไทย สั้น กระชับ สื่อ Intent ลูกค้า
  ตัวอย่าง: "สอบถามสถานะการจัดส่ง", "ขอเปลี่ยน/คืนสินค้า", "ปัญหาการชำระเงิน"
- คำอธิบาย: 1 ประโยคภาษาไทย อธิบาย Pattern ลูกค้ากลุ่มนี้
- ถ้าไม่สามารถระบุได้: ใช้ "ไม่สามารถระบุหัวข้อได้"

ตอบด้วย JSON เท่านั้น ห้ามมีข้อความอื่น:
{{"topic_name": "<ชื่อหัวข้อ>", "topic_description": "<คำอธิบาย>"}}"""


def call_ollama(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = ollama.chat(
                model=OLLAMA_MODEL,
                messages=[{"role": "user", "content": prompt}],
                options={"temperature": 0.0, "num_predict": 200},
            )
            return response["message"]["content"].strip()
        except Exception as e:
            logging.error(f"Ollama error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(5)
    return ""


def parse_json(raw):
    clean = raw.replace("```json", "").replace("```", "").strip()
    try:
        parsed = json.loads(clean)
        if isinstance(parsed, list):
            parsed = parsed[0] if parsed and isinstance(parsed[0], dict) else {}
        return parsed
    except json.JSONDecodeError:
        pass
    start = clean.find("{")
    end   = clean.rfind("}") + 1
    if start != -1 and end > start:
        try:
            return json.loads(clean[start:end])
        except json.JSONDecodeError:
            pass
    logging.warning(f"JSON parse failed. Raw: {raw}")
    return {"error": "parse_failed", "raw": raw}


def label_topics(rep_docs):
    results = {}
    total   = len(rep_docs)
    for i, (topic_id, docs_list) in enumerate(rep_docs.items()):
        logging.info(f"Labeling topic {topic_id} ({i+1}/{total})...")
        truncated = [str(d)[:MAX_DOC_CHARS] for d in docs_list]
        docs_str  = TEXT_DELIMITER.join(truncated)
        raw       = call_ollama(build_prompt(docs_str))
        if not raw:
            results[str(topic_id)] = {"error": "empty_response"}
            continue
        if "ไม่สามารถระบุหัวข้อได้" in raw:
            results[str(topic_id)] = {"topic_name": "ไม่สามารถระบุหัวข้อได้", "topic_description": ""}
        else:
            results[str(topic_id)] = parse_json(raw)
    return results


# ── Run ───────────────────────────────────────────────────────────────────────
topic_labels = label_topics(rep_docs)

# Save labels
labels_path = f"/content/data/topics/labels_{formatted_dt}.json"
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(topic_labels, f, ensure_ascii=False, indent=2)
logging.info(f"Labeled {len(topic_labels)} topics → {labels_path}")

# Merge labels into DataFrame
df["topic_name"] = df["topic_final"].map(
    {int(k): v.get("topic_name", "N/A") for k, v in topic_labels.items()}
)
df["topic_description"] = df["topic_final"].map(
    {int(k): v.get("topic_description", "") for k, v in topic_labels.items()}
)

print(f"✓ Labeled {len(topic_labels)} topics")
df[["raw_message", "topic_final", "topic_name"]].head(5)

## [6/6] Report & Export

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm


def build_summary(df):
    summary = (
        df[df["topic_name"].notna()]
        [df["topic_name"] != "ไม่สามารถระบุหัวข้อได้"]
        .groupby(["topic_name", "topic_description"])
        .size()
        .reset_index(name="message_count")
        .sort_values("message_count", ascending=False)
        .reset_index(drop=True)
    )
    summary["percentage"] = (summary["message_count"] / len(df) * 100).round(1)
    summary.insert(0, "rank", summary.index + 1)
    return summary


def plot_top_topics(summary, top_n=15):
    # Install Thai font for Colab
    try:
        import subprocess
        subprocess.run(["apt-get", "install", "-y", "-q", "fonts-thai-tlwg"], check=False)
        fm._load_fontmanager(try_read_cache=False)
    except Exception:
        pass

    candidates = ["Sarabun", "TH Sarabun New", "Noto Sans Thai", "Garuda", "Waree"]
    thai_font  = None
    for name in candidates:
        matches = [f for f in fm.findSystemFonts() if name.lower() in f.lower()]
        if matches:
            thai_font = fm.FontProperties(fname=matches[0])
            break

    plot_df = summary.head(top_n).sort_values("message_count", ascending=True)
    max_val = plot_df["message_count"].max()

    fig, ax = plt.subplots(figsize=(13, max(6, len(plot_df) * 0.55)))
    bars = ax.barh(plot_df["topic_name"], plot_df["message_count"],
                   color="#1c5872", edgecolor="white", linewidth=0.5)

    for bar, (_, row) in zip(bars, plot_df.iterrows()):
        ax.text(
            bar.get_width() + max_val * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'{int(row["message_count"]):,}  ({row["percentage"]}%)',
            va="center", fontsize=9, color="#333333"
        )

    font_kw = {"fontproperties": thai_font} if thai_font else {}
    ax.set_xlabel("จำนวน Messages", **font_kw)
    ax.set_title("หัวข้อที่ลูกค้าถามมากที่สุด", fontsize=14, pad=15, **font_kw)
    if thai_font:
        for label in ax.get_yticklabels():
            label.set_fontproperties(thai_font)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(0, max_val * 1.18)
    plt.tight_layout()
    plt.savefig("/content/outputs/top_topics.png", dpi=300, bbox_inches="tight")
    logging.info("Chart saved: /content/outputs/top_topics.png")
    plt.show()


# ── Run ───────────────────────────────────────────────────────────────────────
summary = build_summary(df)

df.to_csv("/content/outputs/messages_labeled.csv", index=False, encoding="utf-8-sig")
summary.to_csv("/content/outputs/topic_summary.csv", index=False, encoding="utf-8-sig")
logging.info("Exported: messages_labeled.csv, topic_summary.csv")

plot_top_topics(summary, top_n=TOP_N_CHART)

print("\n" + "═" * 60)
print("  Top Customer Topics by Volume")
print("═" * 60)
print(summary[["rank", "topic_name", "message_count", "percentage"]].to_string(index=False))
print(f"\nTopics identified : {len(summary)}")
print(f"Messages analyzed : {len(df):,}")

## 💾 Download Results

In [ ]:
# ── Download output files to local machine ────────────────────────────────────
from google.colab import files

files.download("/content/outputs/messages_labeled.csv")
files.download("/content/outputs/topic_summary.csv")
files.download("/content/outputs/top_topics.png")
files.download(labels_path)  # labels JSON

print("✓ Download started")